In [1]:
from google.colab import files
uploaded = files.upload()
# cleaned_combined.csv

Saving cleaned_combined.csv to cleaned_combined.csv


In [2]:
import pandas as pd
import numpy as np
import sqlite3

df = pd.read_csv('cleaned_combined.csv')
df['Date'] = pd.to_datetime(df['Date'])

print("Data Loaded!")
print(f"Rows    : {len(df)}")
print(f"Columns : {len(df.columns)}")

Data Loaded!
Rows    : 15368
Columns : 32


In [3]:
# SQLite database
conn = sqlite3.connect('birds.db')

# Data database
df.to_sql('observations', conn, if_exists='replace', index=False)

print("Database created!")
print("Table 'observations' ready!")

Database created!
Table 'observations' ready!


In [9]:
query = """
SELECT
    Location_Type,
    COUNT(*) as Total_Observations,
    COUNT(DISTINCT Common_Name) as Unique_Species,
    ROUND(AVG(Temperature), 2) as Avg_Temperature,
    ROUND(AVG(Humidity), 2) as Avg_Humidity,
    SUM(CASE WHEN PIF_Watchlist_Status = 1 THEN 1 ELSE 0 END) as Watchlist_Sightings
FROM observations
GROUP BY Location_Type
"""
result = pd.read_sql_query(query, conn)
print("=== HABITAT ANALYSIS ===")
print(result.to_string(index=False))

=== HABITAT ANALYSIS ===
Location_Type  Total_Observations  Unique_Species  Avg_Temperature  Avg_Humidity  Watchlist_Sightings
       Forest                8542             108            21.87         77.76                  338
    Grassland                6826             107            23.27         69.66                   40


In [10]:
query = """
SELECT
    Common_Name,
    Scientific_Name,
    COUNT(*) as Sightings,
    COUNT(DISTINCT Admin_Unit_Code) as Parks_Found_In
FROM observations
GROUP BY Common_Name, Scientific_Name
ORDER BY Sightings DESC
LIMIT 10
"""
result = pd.read_sql_query(query, conn)
print("=== TOP 10 SPECIES ===")
print(result.to_string(index=False))

=== TOP 10 SPECIES ===
            Common_Name          Scientific_Name  Sightings  Parks_Found_In
      Northern Cardinal    Cardinalis cardinalis       1125              11
          Carolina Wren Thryothorus ludovicianus        993              11
         Red-eyed Vireo          Vireo olivaceus        737              11
Eastern Tufted Titmouse       Baeolophus bicolor        720              11
         Indigo Bunting         Passerina cyanea        611              10
     Eastern Wood-Pewee          Contopus virens        574              11
          Field Sparrow         Spizella pusilla        492               6
 Red-bellied Woodpecker     Melanerpes carolinus        488              11
         American Robin       Turdus migratorius        470              11
     Acadian Flycatcher      Empidonax virescens        462              11


In [11]:
query = """
SELECT
    Admin_Unit_Code,
    COUNT(*) as Observations,
    COUNT(DISTINCT Common_Name) as Unique_Species,
    SUM(CASE WHEN PIF_Watchlist_Status = 1 THEN 1 ELSE 0 END) as Watchlist_Sightings,
    ROUND(AVG(Temperature), 1) as Avg_Temp
FROM observations
GROUP BY Admin_Unit_Code
ORDER BY Unique_Species DESC
"""
result = pd.read_sql_query(query, conn)
print("=== PARK-WISE DIVERSITY ===")
print(result.to_string(index=False))

=== PARK-WISE DIVERSITY ===
Admin_Unit_Code  Observations  Unique_Species  Watchlist_Sightings  Avg_Temp
           MONO          2522              99                   21      21.4
           MANA          1901              80                   30      24.1
           CHOH          2201              80                   41      20.0
           ANTI          3463              80                   22      23.8
           NACE           683              66                   13      22.1
           PRWI          2462              54                  138      23.4
           HAFE           530              54                   25      22.3
           GWMP           385              49                    6      23.5
           CATO           805              46                   68      21.0
           ROCR           289              45                   14      22.5
           WOTR           127              27                    0      21.5


In [12]:
query = """
SELECT
    Common_Name,
    AOU_Code,
    COUNT(*) as Sightings,
    COUNT(DISTINCT Admin_Unit_Code) as Parks,
    GROUP_CONCAT(DISTINCT Location_Type) as Habitats
FROM observations
WHERE PIF_Watchlist_Status = 1
GROUP BY Common_Name, AOU_Code
ORDER BY Sightings DESC
"""
result = pd.read_sql_query(query, conn)
print("=== WATCHLIST SPECIES ===")
print(result.to_string(index=False))

=== WATCHLIST SPECIES ===
          Common_Name AOU_Code  Sightings  Parks         Habitats
          Wood Thrush     WOTH        309     10 Forest,Grassland
  Worm-eating Warbler     WEWA         31      5           Forest
      Prairie Warbler     PRAW         25      3 Forest,Grassland
     Cerulean Warbler     CERW          7      3           Forest
     Kentucky Warbler     KEWA          2      2 Forest,Grassland
    Willow Flycatcher     WIFL          2      2        Grassland
  Blue-winged Warbler     BWWA          1      1           Forest
Red-headed Woodpecker     RHWO          1      1           Forest


In [13]:
query = """
SELECT
    Month,
    Month_Name,
    Season,
    COUNT(*) as Observations,
    COUNT(DISTINCT Common_Name) as Unique_Species
FROM observations
GROUP BY Month, Month_Name, Season
ORDER BY Month
"""
result = pd.read_sql_query(query, conn)
print("=== MONTHLY TREND ===")
print(result.to_string(index=False))

=== MONTHLY TREND ===
 Month Month_Name Season  Observations  Unique_Species
     5        May Spring          4863             113
     6       June Summer          6209              99
     7       July Summer          4296              88


In [14]:
query = """
SELECT
    Observer,
    COUNT(*) as Observations,
    COUNT(DISTINCT Common_Name) as Unique_Species,
    COUNT(DISTINCT Admin_Unit_Code) as Parks_Covered
FROM observations
GROUP BY Observer
ORDER BY Observations DESC
"""
result = pd.read_sql_query(query, conn)
print("=== OBSERVER ANALYSIS ===")
print(result.to_string(index=False))

=== OBSERVER ANALYSIS ===
        Observer  Observations  Unique_Species  Parks_Covered
Elizabeth Oswald          5759             119             11
  Kimberly Serno          5346              90             11
  Brian Swimelar          4263              83             10


In [15]:
query = """
SELECT
    Sex,
    COUNT(*) as Count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM observations), 1) as Percentage
FROM observations
GROUP BY Sex
ORDER BY Count DESC
"""
result = pd.read_sql_query(query, conn)
print("=== SEX DISTRIBUTION ===")
print(result.to_string(index=False))

=== SEX DISTRIBUTION ===
         Sex  Count  Percentage
Undetermined  12133        78.9
        Male   3109        20.2
      Female    126         0.8


In [16]:
query = """
SELECT
    Sky,
    COUNT(*) as Observations,
    ROUND(AVG(Temperature), 1) as Avg_Temp,
    ROUND(AVG(Humidity), 1) as Avg_Humidity
FROM observations
GROUP BY Sky
ORDER BY Observations DESC
"""
result = pd.read_sql_query(query, conn)
print("=== SKY CONDITIONS ===")
print(result.to_string(index=False))

=== SKY CONDITIONS ===
                Sky  Observations  Avg_Temp  Avg_Humidity
      Partly Cloudy          6172      23.0          73.5
Clear or Few Clouds          5330      22.8          72.9
    Cloudy/Overcast          2916      21.7          77.3
                Fog           598      20.1          76.7
       Mist/Drizzle           352      21.3          74.6


In [17]:
conn.close()
print("SQL Analysis Complete!")

from google.colab import files
files.download('birds.db')

SQL Analysis Complete!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>